In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("../data/raw/online_shoppers_intention.csv")
print(df.shape)

(12330, 18)


In [4]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
def add_engineered_features(data):
    data = data.copy()

    data["TotalPages"] = (
        data["Administrative"]
        + data["Informational"]
        + data["ProductRelated"]
    )

    data["TotalDuration"] = (
        data["Administrative_Duration"]
        + data["Informational_Duration"]
        + data["ProductRelated_Duration"]
    )

    data["AvgDurationPerPage"] = np.where(
        data["TotalPages"] > 0,
        data["TotalDuration"] / data["TotalPages"],
        0
    )

    data["ProductPageShare"] = np.where(
        data["TotalPages"] > 0,
        data["ProductRelated"] / data["TotalPages"],
        0
    )

    data["ProductDurationShare"] = np.where(
        data["TotalDuration"] > 0,
        data["ProductRelated_Duration"] / data["TotalDuration"],
        0
    )

    return data

In [7]:
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

In [8]:
engineered_features = [
    "TotalPages",
    "TotalDuration",
    "AvgDurationPerPage",
    "ProductPageShare",
    "ProductDurationShare"
]

print(X_train[engineered_features].isna().sum())
print(np.isinf(X_train[engineered_features]).sum())

TotalPages              0
TotalDuration           0
AvgDurationPerPage      0
ProductPageShare        0
ProductDurationShare    0
dtype: int64
TotalPages              0
TotalDuration           0
AvgDurationPerPage      0
ProductPageShare        0
ProductDurationShare    0
dtype: int64


In [9]:
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

boolean_features = ["Weekend"]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + boolean_features
]

In [10]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)

In [11]:
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features),
        ("bool", "passthrough", boolean_features)
    ]
)

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)

In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [15]:
def summarize_cv(model_name, pipeline, X_data, y_data):

    scores = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    row = {"Model": model_name}

    for metric in scoring:
        row[f"CV_{metric}_mean"] = (
            scores[f"test_{metric}"].mean()
        )

        row[f"CV_{metric}_std"] = (
            scores[f"test_{metric}"].std()
        )

    row["Train_F1_mean"] = scores["train_f1"].mean()
    row["Validation_F1_mean"] = scores["test_f1"].mean()

    return row, scores

In [16]:
from sklearn.dummy import DummyClassifier

In [17]:
dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", DummyClassifier(
        strategy="most_frequent"
    ))
])

In [18]:
dummy_row, dummy_scores = summarize_cv(
    "Dummy - Most Frequent",
    dummy_pipeline,
    X_train,
    y_train
)

pd.DataFrame([dummy_row])

,Model,CV_accuracy_mean,CV_accuracy_std,CV_precision_mean,CV_precision_std,CV_recall_mean,CV_recall_std,CV_f1_mean,CV_f1_std,CV_roc_auc_mean,CV_roc_auc_std,CV_pr_auc_mean,CV_pr_auc_std,Train_F1_mean,Validation_F1_mean
0,Dummy - Most Frequent,0.845296,0.000197,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.154704,0.000197,0.0,0.0


In [23]:
#KNN baseline
from sklearn.neighbors import KNeighborsClassifier

knn_baseline = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", KNeighborsClassifier(
        n_neighbors=5
    ))
])

knn_baseline_row, knn_baseline_scores = summarize_cv(
    "KNN - Baseline",
    knn_baseline,
    X_train,
    y_train
)

pd.DataFrame([knn_baseline_row])

,Model,CV_accuracy_mean,CV_accuracy_std,CV_precision_mean,CV_precision_std,CV_recall_mean,CV_recall_std,CV_f1_mean,CV_f1_std,CV_roc_auc_mean,CV_roc_auc_std,CV_pr_auc_mean,CV_pr_auc_std,Train_F1_mean,Validation_F1_mean
0,KNN - Baseline,0.873885,0.003874,0.659136,0.021631,0.382704,0.018907,0.484085,0.019386,0.797097,0.007882,0.492921,0.016482,0.611721,0.484085


In [25]:
knn_tune_pipeline = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", KNeighborsClassifier())
])

knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_search = GridSearchCV(
    estimator=knn_tune_pipeline,
    param_grid=knn_param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=1,
    refit=True
)

knn_search.fit(X_train, y_train)

print("Best CV PR-AUC:", knn_search.best_score_)
print("Best parameters:", knn_search.best_params_)

best_knn = knn_search.best_estimator_

Best CV PR-AUC: 0.6217437552316329
Best parameters: {'model__n_neighbors': 31, 'model__p': 2, 'model__weights': 'distance'}


In [26]:
knn_tuned_row, knn_tuned_scores = summarize_cv(
    "KNN - Tuned",
    best_knn,
    X_train,
    y_train
)

knn_compare = pd.DataFrame([
    knn_baseline_row,
    knn_tuned_row
])

knn_compare[
    [
        "Model",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean"
    ]
]

,Model,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean
0,KNN - Baseline,0.659136,0.382704,0.484085,0.797097,0.492921
1,KNN - Tuned,0.801753,0.300793,0.436905,0.864157,0.621742
